# 02 — Làm sạch dữ liệu và tạo biến phân tích

Notebook này chuẩn hóa 48 file NYC 311 giai đoạn **2022–2025**, kiểm tra chất lượng dữ liệu và tạo các biến dùng cho EDA/dashboard.

**Quy ước quan trọng về thời gian đóng yêu cầu**

- Ngày chốt dữ liệu (`SNAPSHOT_DATE`) là **06/10/2026 23:59:59**.
- Một closure chỉ được coi là hợp lệ khi `closed_date` tồn tại, không trước `created_date`, **và không vượt quá ngày chốt dữ liệu**.
- `closure_hours` là **thời gian đóng yêu cầu được ghi nhận trong hệ thống**, không đồng nghĩa với thời gian xử lý vật lý ngoài thực tế.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.6
NumPy: 2.5.3


In [3]:
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

# Ngày chốt dữ liệu dùng nhất quán cho feature engineering và EDA
SNAPSHOT_DATE = pd.Timestamp("2026-10-06 23:59:59")
print("Snapshot date :", SNAPSHOT_DATE)

Project root: d:\nyc311_analysis
Raw: d:\nyc311_analysis\data\raw
Processed: d:\nyc311_analysis\data\processed
Snapshot date : 2026-10-06 23:59:59


## 1. Kiểm tra dữ liệu đầu vào

Trước khi làm sạch, notebook kiểm tra đủ 48 partition theo tháng, số dòng và schema. Việc này giúp phát hiện file thiếu/thừa cột hoặc sai thứ tự cột trước khi xử lý hàng loạt.

In [4]:
raw_files = sorted(
    RAW_DIR.glob("nyc311_*_*_raw.parquet")
)

print("Number of Parquet files:", len(raw_files))

for file in raw_files[:5]:
    print(file.name)

print("...")

for file in raw_files[-5:]:
    print(file.name)

Number of Parquet files: 48
nyc311_2022_01_raw.parquet
nyc311_2022_02_raw.parquet
nyc311_2022_03_raw.parquet
nyc311_2022_04_raw.parquet
nyc311_2022_05_raw.parquet
...
nyc311_2025_08_raw.parquet
nyc311_2025_09_raw.parquet
nyc311_2025_10_raw.parquet
nyc311_2025_11_raw.parquet
nyc311_2025_12_raw.parquet


In [5]:
records = []

for file in raw_files:
    parquet_file = pq.ParquetFile(file)

    parts = file.stem.split("_")

    year = int(parts[1])
    month = int(parts[2])

    records.append({
        "file": file.name,
        "year": year,
        "month": month,
        "rows": parquet_file.metadata.num_rows
    })

file_summary = pd.DataFrame(records)

display(file_summary.head())
year_summary = (
    file_summary
    .groupby("year", as_index=False)["rows"]
    .sum()
)

display(year_summary)

print(
    "TOTAL:",
    f"{file_summary['rows'].sum():,}"
)

,file,year,month,rows
0,nyc311_2022_01_raw.parquet,2022,1,279281
1,nyc311_2022_02_raw.parquet,2022,2,244234
2,nyc311_2022_03_raw.parquet,2022,3,251933
3,nyc311_2022_04_raw.parquet,2022,4,236083
4,nyc311_2022_05_raw.parquet,2022,5,276664


,year,rows
0,2022,3169960
1,2023,3224723
2,2024,3456770
3,2025,3655041


TOTAL: 13,506,494


In [6]:
month_check = (
    file_summary
    .groupby("year")["month"]
    .nunique()
)

display(month_check)

year
2022    12
2023    12
2024    12
2025    12
Name: month, dtype: int64

In [7]:
schemas = {}

for file in raw_files:
    schema = pq.read_schema(file)

    schemas[file.name] = [
        field.name
        for field in schema
    ]

reference_columns = schemas[raw_files[0].name]

schema_errors = []

for filename, columns in schemas.items():
    if columns != reference_columns:
        schema_errors.append(filename)

print("Number of columns:", len(reference_columns))
print("Columns:")
print(reference_columns)

print()
print(
    "Files with different schema:",
    len(schema_errors)
)

Number of columns: 15
Columns:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'community_board', 'borough', 'status', 'open_data_channel_type', 'latitude', 'longitude']

Files with different schema: 2


In [8]:
EXPECTED_COLUMN_ORDER = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "community_board",
    "borough",
    "status",
    "open_data_channel_type",
    "latitude",
    "longitude"
]

EXPECTED_COLUMNS = set(EXPECTED_COLUMN_ORDER)

missing_column_files = []
extra_column_files = []
order_only_files = []

for file in raw_files:
    schema = pq.read_schema(file)

    actual_columns = [
        field.name
        for field in schema
    ]

    actual_set = set(actual_columns)

    missing = EXPECTED_COLUMNS - actual_set
    extra = actual_set - EXPECTED_COLUMNS

    if missing:
        missing_column_files.append({
            "file": file.name,
            "missing": sorted(missing)
        })

    if extra:
        extra_column_files.append({
            "file": file.name,
            "extra": sorted(extra)
        })

    if (
        not missing
        and not extra
        and actual_columns != EXPECTED_COLUMN_ORDER
    ):
        order_only_files.append(file.name)

print("Files missing columns :", len(missing_column_files))
print("Files with extra cols :", len(extra_column_files))
print("Order different only  :", len(order_only_files))

if missing_column_files:
    print("\nMISSING:")
    for x in missing_column_files:
        print(x)

if extra_column_files:
    print("\nEXTRA:")
    for x in extra_column_files:
        print(x)

Files missing columns : 0
Files with extra cols : 0
Order different only  : 2


## 2. Khảo sát chất lượng dữ liệu

Thực hiện profiling trên một partition mẫu và toàn bộ 48 file: missing value, placeholder dạng văn bản (`N/A`, `Unspecified`, ...), và lỗi chuyển kiểu ngày/tọa độ. `UNKNOWN` của `open_data_channel_type` được **giữ như một nhóm nguồn hợp lệ**, không tự động đổi thành missing.

In [9]:
sample_file = RAW_DIR / "nyc311_2025_01_raw.parquet"

df = pd.read_parquet(sample_file)

df = df.reindex(
    columns=EXPECTED_COLUMN_ORDER
)

print("Shape:", df.shape)

display(df.head())

Shape: (348180, 15)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,community_board,borough,status,open_data_channel_type,latitude,longitude
0,63577994,2025-01-01T00:00:12.000,2025-01-02T17:05:54.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10466,12 BRONX,BRONX,Closed,MOBILE,40.89187241649303,-73.86016845296459
1,63584096,2025-01-01T00:00:13.000,2025-01-01T00:05:19.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,11211,01 BROOKLYN,BROOKLYN,Closed,MOBILE,40.71120965026345,-73.95897331500413
2,63581480,2025-01-01T00:00:17.000,2025-01-01T09:01:06.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10466,12 BRONX,BRONX,Closed,MOBILE,40.89187241649303,-73.86016845296459
3,63575612,2025-01-01T00:00:19.000,2025-01-01T00:43:08.000,NYPD,New York City Police Department,Illegal Fireworks,N/A,Street/Sidewalk,11230,14 BROOKLYN,BROOKLYN,Closed,ONLINE,40.62923998368745,-73.96879043074365
4,63586536,2025-01-01T00:00:34.000,2025-01-02T12:04:30.000,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,APARTMENT ONLY,RESIDENTIAL BUILDING,10466,12 BRONX,BRONX,Closed,ONLINE,40.89194007532644,-73.85277542800861


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 348180 entries, 0 to 348179
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   unique_key              348180 non-null  string
 1   created_date            348180 non-null  str   
 2   closed_date             345583 non-null  str   
 3   agency                  348180 non-null  str   
 4   agency_name             348180 non-null  str   
 5   complaint_type          348180 non-null  str   
 6   descriptor              346641 non-null  str   
 7   location_type           313924 non-null  str   
 8   incident_zip            345845 non-null  str   
 9   community_board         348180 non-null  str   
 10  borough                 348180 non-null  str   
 11  status                  348180 non-null  str   
 12  open_data_channel_type  348180 non-null  str   
 13  latitude                345075 non-null  str   
 14  longitude               345075 non-null  str   

In [11]:
missing_profile = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (
        df.isna().mean() * 100
    ).round(2)
})

missing_profile = missing_profile.sort_values(
    "missing_percent",
    ascending=False
)

display(missing_profile)

,missing_count,missing_percent
location_type,34256,9.84
longitude,3105,0.89
latitude,3105,0.89
closed_date,2597,0.75
incident_zip,2335,0.67
descriptor,1539,0.44
unique_key,0,0.00
complaint_type,0,0.00
agency_name,0,0.00
agency,0,0.00


In [12]:
categorical_cols = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "community_board",
    "borough",
    "status",
    "open_data_channel_type"
]

cardinality = []

for col in categorical_cols:
    cardinality.append({
        "column": col,
        "unique_values": df[col].nunique(dropna=False)
    })

cardinality_df = pd.DataFrame(cardinality)

display(cardinality_df)

,column,unique_values
0,agency,14
1,agency_name,14
2,complaint_type,168
3,descriptor,757
4,location_type,111
5,incident_zip,239
6,community_board,77
7,borough,6
8,status,7
9,open_data_channel_type,5


In [13]:
PLACEHOLDERS = [
    "N/A",
    "NA",
    "UNKNOWN",
    "Unknown",
    "Unspecified",
    "UNSPECIFIED"
]

placeholder_counts = []

for col in categorical_cols:
    if df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col]):
        values = (
            df[col]
            .astype("string")
            .str.strip()
        )

        for placeholder in PLACEHOLDERS:
            count = (values == placeholder).sum()

            if count > 0:
                placeholder_counts.append({
                    "column": col,
                    "value": placeholder,
                    "count": int(count),
                    "percent": round(count / len(df) * 100, 2)
                })

placeholder_df = pd.DataFrame(placeholder_counts)

display(placeholder_df)

,column,value,count,percent
0,complaint_type,Unspecified,1,0.00
1,descriptor,N/A,4162,1.20
2,descriptor,Unspecified,1,0.00
3,location_type,N/A,8,0.00
4,location_type,Unspecified,1,0.00
5,borough,Unspecified,177,0.05
6,status,Unspecified,36,0.01
7,open_data_channel_type,UNKNOWN,20683,5.94


In [14]:
# profile missing and placeholder
from collections import defaultdict
import pandas as pd

PLACEHOLDERS = [
    "N/A",
    "NA",
    "UNKNOWN",
    "Unknown",
    "Unspecified",
    "UNSPECIFIED",
    ""
]

PROFILE_COLUMNS = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "community_board",
    "borough",
    "status",
    "open_data_channel_type"
]

total_rows = 0

missing_total = pd.Series(
    0,
    index=EXPECTED_COLUMN_ORDER,
    dtype="int64"
)

placeholder_total = defaultdict(int)

conversion_errors = {
    "created_date": 0,
    "closed_date": 0,
    "latitude": 0,
    "longitude": 0
}

for i, file in enumerate(raw_files, start=1):

    part = pd.read_parquet(file)

    part = part.reindex(
        columns=EXPECTED_COLUMN_ORDER
    )

    total_rows += len(part)

    # 1. Missing thật
    missing_total += part.isna().sum().astype("int64")

    # 2. Placeholder text
    for col in PROFILE_COLUMNS:

        values = (
            part[col]
            .astype("string")
            .str.strip()
        )

        for placeholder in PLACEHOLDERS:

            count = (values == placeholder).sum()

            if count > 0:
                placeholder_total[
                    (col, placeholder)
                ] += int(count)

    # 3. Date conversion test
    created = pd.to_datetime(
        part["created_date"],
        errors="coerce"
    )

    conversion_errors["created_date"] += int(
        (
            part["created_date"].notna()
            & created.isna()
        ).sum()
    )

    closed = pd.to_datetime(
        part["closed_date"],
        errors="coerce"
    )

    conversion_errors["closed_date"] += int(
        (
            part["closed_date"].notna()
            & closed.isna()
        ).sum()
    )

    # 4. Coordinate conversion test
    lat = pd.to_numeric(
        part["latitude"],
        errors="coerce"
    )

    lon = pd.to_numeric(
        part["longitude"],
        errors="coerce"
    )

    conversion_errors["latitude"] += int(
        (
            part["latitude"].notna()
            & lat.isna()
        ).sum()
    )

    conversion_errors["longitude"] += int(
        (
            part["longitude"].notna()
            & lon.isna()
        ).sum()
    )

    print(
        f"[{i:02d}/{len(raw_files)}] "
        f"{file.name} done"
    )

print("\nFULL PROFILING COMPLETE")
print("Total rows:", f"{total_rows:,}")


[01/48] nyc311_2022_01_raw.parquet done
[02/48] nyc311_2022_02_raw.parquet done
[03/48] nyc311_2022_03_raw.parquet done
[04/48] nyc311_2022_04_raw.parquet done
[05/48] nyc311_2022_05_raw.parquet done
[06/48] nyc311_2022_06_raw.parquet done
[07/48] nyc311_2022_07_raw.parquet done
[08/48] nyc311_2022_08_raw.parquet done
[09/48] nyc311_2022_09_raw.parquet done
[10/48] nyc311_2022_10_raw.parquet done
[11/48] nyc311_2022_11_raw.parquet done
[12/48] nyc311_2022_12_raw.parquet done
[13/48] nyc311_2023_01_raw.parquet done
[14/48] nyc311_2023_02_raw.parquet done
[15/48] nyc311_2023_03_raw.parquet done
[16/48] nyc311_2023_04_raw.parquet done
[17/48] nyc311_2023_05_raw.parquet done
[18/48] nyc311_2023_06_raw.parquet done
[19/48] nyc311_2023_07_raw.parquet done
[20/48] nyc311_2023_08_raw.parquet done
[21/48] nyc311_2023_09_raw.parquet done
[22/48] nyc311_2023_10_raw.parquet done
[23/48] nyc311_2023_11_raw.parquet done
[24/48] nyc311_2023_12_raw.parquet done
[25/48] nyc311_2024_01_raw.parquet done


In [15]:
full_missing_profile = pd.DataFrame({
    "missing_count": missing_total,
    "missing_percent": (
        missing_total / total_rows * 100
    ).round(2)
})

full_missing_profile = (
    full_missing_profile
    .sort_values(
        "missing_percent",
        ascending=False
    )
)

display(full_missing_profile)

,missing_count,missing_percent
location_type,1644216,12.17
closed_date,205276,1.52
longitude,202044,1.50
latitude,202044,1.50
incident_zip,147171,1.09
descriptor,122494,0.91
unique_key,0,0.00
complaint_type,0,0.00
agency_name,0,0.00
agency,0,0.00


In [16]:
raw_duplicate_total = 0

for file in raw_files:
    keys = pd.read_parquet(
        file,
        columns=["unique_key"]
    )["unique_key"].astype("string").str.strip()

    raw_duplicate_total += int(
        keys.duplicated().sum()
    )

print("Số phân vùng raw:", len(raw_files))
print("Số unique_key trùng trong từng phân vùng:",
      raw_duplicate_total)

Số phân vùng raw: 48
Số unique_key trùng trong từng phân vùng: 0


In [17]:
placeholder_rows = []

for (column, value), count in placeholder_total.items():

    placeholder_rows.append({
        "column": column,
        "value": value,
        "count": count,
        "percent": round(
            count / total_rows * 100,
            2
        )
    })

full_placeholder_profile = pd.DataFrame(
    placeholder_rows
)

full_placeholder_profile = (
    full_placeholder_profile
    .sort_values(
        ["column", "count"],
        ascending=[True, False]
    )
)

display(full_placeholder_profile)

,column,value,count,percent
2,borough,Unspecified,15065,0.11
8,complaint_type,Unspecified,42,0.00
0,descriptor,N/A,269802,2.00
9,descriptor,Unspecified,42,0.00
5,incident_zip,NA,4,0.00
6,incident_zip,N/A,2,0.00
1,location_type,N/A,883,0.01
10,location_type,Unspecified,42,0.00
7,location_type,Unknown,1,0.00
4,open_data_channel_type,UNKNOWN,1072807,7.94


In [18]:
conversion_profile = pd.DataFrame({
    "column": conversion_errors.keys(),
    "conversion_errors": conversion_errors.values()
})

display(conversion_profile)

,column,conversion_errors
0,created_date,0
1,closed_date,0
2,latitude,0
3,longitude,0


## 3. Hàm làm sạch

`clean_partition()` chỉ xử lý schema, chuỗi, placeholder và kiểu dữ liệu. Các biến dẫn xuất như `closure_valid` và `closure_hours` được tạo ở bước **feature engineering**, để tách rõ cleaning khỏi logic phân tích.

In [19]:
#cleaning
def clean_partition(df):
    df = df.copy()

    # 1. Chuẩn hóa thứ tự cột
    df = df.reindex(
        columns=EXPECTED_COLUMN_ORDER
    )

    # 2. Chuẩn hóa string cơ bản
    string_cols = [
        "unique_key",
        "agency",
        "agency_name",
        "complaint_type",
        "descriptor",
        "location_type",
        "incident_zip",
        "community_board",
        "borough",
        "status",
        "open_data_channel_type"
    ]

    for col in string_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
        )

    # 3. Placeholder -> missing
    df["complaint_type"] = df["complaint_type"].replace({
        "Unspecified": pd.NA
    })

    df["descriptor"] = df["descriptor"].replace({
        "N/A": pd.NA,
        "Unspecified": pd.NA
    })

    df["location_type"] = df["location_type"].replace({
        "N/A": pd.NA,
        "Unknown": pd.NA,
        "Unspecified": pd.NA
    })

    df["incident_zip"] = df["incident_zip"].replace({
        "N/A": pd.NA,
        "NA": pd.NA
    })

    df["borough"] = df["borough"].replace({
        "Unspecified": pd.NA
    })

    df["status"] = df["status"].replace({
        "Unspecified": pd.NA
    })

    # 4. Chuẩn hóa categorical quan trọng
    df["borough"] = df["borough"].str.upper()

    df["open_data_channel_type"] = (
        df["open_data_channel_type"]
        .str.upper()
    )

    # UNKNOWN ở channel được giữ nguyên

    # 5. Convert datatype
    df["created_date"] = pd.to_datetime(
        df["created_date"],
        errors="coerce"
    )

    df["closed_date"] = pd.to_datetime(
        df["closed_date"],
        errors="coerce"
    )

    df["latitude"] = pd.to_numeric(
        df["latitude"],
        errors="coerce"
    )

    df["longitude"] = pd.to_numeric(
        df["longitude"],
        errors="coerce"
    )

    return df

In [20]:
df_clean = clean_partition(df)

print(df_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 348180 entries, 0 to 348179
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   unique_key              348180 non-null  string        
 1   created_date            348180 non-null  datetime64[us]
 2   closed_date             345583 non-null  datetime64[us]
 3   agency                  348180 non-null  string        
 4   agency_name             348180 non-null  string        
 5   complaint_type          348179 non-null  string        
 6   descriptor              342478 non-null  string        
 7   location_type           313915 non-null  string        
 8   incident_zip            345845 non-null  string        
 9   community_board         348180 non-null  string        
 10  borough                 348003 non-null  string        
 11  status                  348144 non-null  string        
 12  open_data_channel_type  348180 non-null  

In [21]:
print("Shape before:", df.shape)
print("Shape after :", df_clean.shape)

print(
    "Duplicate unique_key:",
    df_clean["unique_key"]
    .duplicated()
    .sum()
)

print(
    "Missing created_date:",
    df_clean["created_date"]
    .isna()
    .sum()
)

Shape before: (348180, 15)
Shape after : (348180, 15)
Duplicate unique_key: 0
Missing created_date: 0


In [22]:
check_cols = [
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "borough",
    "status",
    "open_data_channel_type"
]

for col in check_cols:
    print(f"\n--- {col} ---")

    print(
        df_clean[col]
        .value_counts(
            dropna=False
        )
        .head(10)
    )


--- complaint_type ---
complaint_type
Noise - Residential     70837
HEAT/HOT WATER          67229
Illegal Parking         43139
Blocked Driveway        14661
UNSANITARY CONDITION    10000
PLUMBING                 7409
Water System             6669
Abandoned Vehicle        5626
PAINT/PLASTER            5588
Street Condition         5474
Name: count, dtype: int64[pyarrow]

--- descriptor ---


descriptor
Loud Music/Party                 62325
ENTIRE BUILDING                  42134
APARTMENT ONLY                   25095
Banging/Pounding                 12358
Blocked Hydrant                  11805
No Access                        10770
Posted Parking Sign Violation    10078
Blocked Sidewalk                  6591
<NA>                              5702
With License Plate                5626
Name: count, dtype: int64[pyarrow]

--- location_type ---
location_type
RESIDENTIAL BUILDING          110624
Street/Sidewalk                74306
Residential Building/House     73203
<NA>                           34265
Street                         19584
Sidewalk                       14506
Store/Commercial                3476
Club/Bar/Restaurant             1959
Above Address                   1815
3+ Family Apt. Building         1403
Name: count, dtype: int64[pyarrow]

--- incident_zip ---
incident_zip
10466    48604
10467     5599
11226     5251
10458     5008
10452     4974
10453     48

In [23]:
checks = {
    "descriptor N/A": (
        df_clean["descriptor"] == "N/A"
    ).sum(),

    "descriptor Unspecified": (
        df_clean["descriptor"] == "Unspecified"
    ).sum(),

    "location_type N/A": (
        df_clean["location_type"] == "N/A"
    ).sum(),

    "borough Unspecified": (
        df_clean["borough"] == "Unspecified"
    ).sum(),

    "status Unspecified": (
        df_clean["status"] == "Unspecified"
    ).sum(),

    "channel UNKNOWN": (
        df_clean["open_data_channel_type"]
        == "UNKNOWN"
    ).sum()
}

pd.Series(checks)
# Kiểm tra closed_date có trước created_date không
closed_before_created = (
    df_clean["closed_date"].notna()
    &
    (
        df_clean["closed_date"]
        < df_clean["created_date"]
    )
)

print(
    "Closed before created:",
    closed_before_created.sum()
)

display(
    df_clean.loc[
        closed_before_created,
        [
            "unique_key",
            "created_date",
            "closed_date",
            "agency",
            "complaint_type",
            "status"
        ]
    ].head(10)
)

Closed before created: 99


,unique_key,created_date,closed_date,agency,complaint_type,status
9178,63591237,2025-01-01 19:07:52,2024-12-04 09:33:00,DOT,Sidewalk Condition,Closed
9644,63592286,2025-01-01 20:19:16,2024-12-04 09:33:00,DOT,Sidewalk Condition,Closed
28246,63622430,2025-01-03 09:07:00,2025-01-02 08:59:00,DOT,Street Light Condition,Pending
64138,63656893,2025-01-06 07:51:00,2025-01-04 02:00:00,DOT,Street Light Condition,Pending
109613,63694583,2025-01-09 11:05:00,2025-01-03 11:05:00,DOT,Street Light Condition,Pending
109635,63694823,2025-01-09 11:06:00,2025-01-03 11:06:00,DOT,Street Light Condition,Pending
109666,63692739,2025-01-09 11:07:00,2025-01-03 11:07:00,DOT,Street Light Condition,Pending
110499,63702502,2025-01-09 12:12:00,2025-01-06 12:12:00,DOT,Street Light Condition,Pending
110701,63697586,2025-01-09 12:26:00,2025-01-07 12:26:00,DOT,Street Light Condition,Pending
110748,63695071,2025-01-09 12:30:00,2025-01-07 22:30:00,DOT,Street Light Condition,Pending


In [24]:
# Kiểm tra closed_date có trước created_date không
closed_before_created = (
    df_clean["closed_date"].notna()
    &
    (
        df_clean["closed_date"]
        < df_clean["created_date"]
    )
)

print(
    "Closed before created:",
    closed_before_created.sum()
)

display(
    df_clean.loc[
        closed_before_created,
        [
            "unique_key",
            "created_date",
            "closed_date",
            "agency",
            "complaint_type",
            "status"
        ]
    ].head(10)
)

Closed before created: 99


,unique_key,created_date,closed_date,agency,complaint_type,status
9178,63591237,2025-01-01 19:07:52,2024-12-04 09:33:00,DOT,Sidewalk Condition,Closed
9644,63592286,2025-01-01 20:19:16,2024-12-04 09:33:00,DOT,Sidewalk Condition,Closed
28246,63622430,2025-01-03 09:07:00,2025-01-02 08:59:00,DOT,Street Light Condition,Pending
64138,63656893,2025-01-06 07:51:00,2025-01-04 02:00:00,DOT,Street Light Condition,Pending
109613,63694583,2025-01-09 11:05:00,2025-01-03 11:05:00,DOT,Street Light Condition,Pending
109635,63694823,2025-01-09 11:06:00,2025-01-03 11:06:00,DOT,Street Light Condition,Pending
109666,63692739,2025-01-09 11:07:00,2025-01-03 11:07:00,DOT,Street Light Condition,Pending
110499,63702502,2025-01-09 12:12:00,2025-01-06 12:12:00,DOT,Street Light Condition,Pending
110701,63697586,2025-01-09 12:26:00,2025-01-07 12:26:00,DOT,Street Light Condition,Pending
110748,63695071,2025-01-09 12:30:00,2025-01-07 22:30:00,DOT,Street Light Condition,Pending


In [25]:
# ==============================
# DATA VALIDATION - JAN 2025
# ==============================

# 1. Closed date trước Created date
invalid_closure = (
    df_clean["closed_date"].notna()
    & df_clean["created_date"].notna()
    & (df_clean["closed_date"] < df_clean["created_date"])
)

# 2. Closed date vượt quá ngày chốt dữ liệu
future_closure = (
    df_clean["closed_date"].notna()
    & (df_clean["closed_date"] > SNAPSHOT_DATE)
)

# 3. Borough hợp lệ
VALID_BOROUGHS = {
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
}

invalid_borough = (
    df_clean["borough"].notna()
    & ~df_clean["borough"].isin(VALID_BOROUGHS)
)

# 4. Latitude / Longitude ngoài giới hạn địa lý
invalid_latitude = (
    df_clean["latitude"].notna()
    & ~df_clean["latitude"].between(-90, 90)
)

invalid_longitude = (
    df_clean["longitude"].notna()
    & ~df_clean["longitude"].between(-180, 180)
)

# 5. ZIP không có đúng 5 chữ số
zip_values = df_clean["incident_zip"].dropna()
invalid_zip = ~zip_values.str.fullmatch(r"\d{5}")

# 6. Status Closed nhưng thiếu closed_date
closed_without_date = (
    (df_clean["status"] == "Closed")
    & df_clean["closed_date"].isna()
)

validation_summary = pd.DataFrame({
    "check": [
        "Closed date before created date",
        "Closed date after snapshot",
        "Invalid borough",
        "Invalid latitude",
        "Invalid longitude",
        "ZIP not 5 digits",
        "Status Closed but no closed_date"
    ],
    "count": [
        int(invalid_closure.sum()),
        int(future_closure.sum()),
        int(invalid_borough.sum()),
        int(invalid_latitude.sum()),
        int(invalid_longitude.sum()),
        int(invalid_zip.sum()),
        int(closed_without_date.sum())
    ]
})

display(validation_summary)

print("\nLatitude range:")
print(df_clean["latitude"].min(), "→", df_clean["latitude"].max())

print("\nLongitude range:")
print(df_clean["longitude"].min(), "→", df_clean["longitude"].max())

print("\nStatus of rows with missing closed_date:")
display(
    df_clean.loc[
        df_clean["closed_date"].isna(),
        "status"
    ].value_counts(dropna=False)
)

,check,count
0,Closed date before created date,99
1,Closed date after snapshot,0
2,Invalid borough,0
3,Invalid latitude,0
4,Invalid longitude,0
5,ZIP not 5 digits,0
6,Status Closed but no closed_date,1051



Latitude range:
40.498948846168354 → 40.91229473631874

Longitude range:
-74.25432584765962 → -73.70072272310216

Status of rows with missing closed_date:


status
Closed         1051
In Progress     859
Open            343
Assigned        184
Pending          85
Started          74
<NA>              1
Name: count, dtype: int64[pyarrow]

## 4. Feature engineering

Tạo 8 biến phân tích: `year`, `month`, `weekday`, `hour`, `is_weekend`, `closure_valid`, `closure_hours`, `has_coordinates`.

Điểm sửa quan trọng ở phiên bản này: `closure_valid` dùng cùng một quy tắc với EDA/dashboard, bao gồm điều kiện **`closed_date <= SNAPSHOT_DATE`**.

In [26]:
# ==========================================
# FEATURE ENGINEERING - JANUARY 2025
# ==========================================

df_featured = df_clean.copy()

# ----- Time features -----
df_featured["year"] = df_featured["created_date"].dt.year
df_featured["month"] = df_featured["created_date"].dt.month
df_featured["weekday"] = df_featured["created_date"].dt.day_name()
df_featured["hour"] = df_featured["created_date"].dt.hour
df_featured["is_weekend"] = (
    df_featured["created_date"].dt.dayofweek >= 5
)

# ----- Closure validation -----
df_featured["closure_valid"] = (
    df_featured["created_date"].notna()
    & df_featured["closed_date"].notna()
    & (df_featured["closed_date"] >= df_featured["created_date"])
    & (df_featured["closed_date"] <= SNAPSHOT_DATE)
)

# Chỉ tính closure_hours khi closure hợp lệ
df_featured["closure_hours"] = np.nan

valid_mask = df_featured["closure_valid"]

df_featured.loc[
    valid_mask,
    "closure_hours"
] = (
    (
        df_featured.loc[valid_mask, "closed_date"]
        - df_featured.loc[valid_mask, "created_date"]
    )
    .dt.total_seconds()
    / 3600
)

df_featured["closure_hours"] = pd.to_numeric(
    df_featured["closure_hours"],
    errors="coerce"
)

# ----- Geographic availability -----
df_featured["has_coordinates"] = (
    df_featured["latitude"].notna()
    & df_featured["longitude"].notna()
)

print("Shape:", df_featured.shape)
print("Valid closure records:", int(df_featured["closure_valid"].sum()))
print(
    "Invalid/unavailable closure:",
    int((~df_featured["closure_valid"]).sum())
)
print(
    "Future closed_date excluded:",
    int(
        (
            df_featured["closed_date"].notna()
            & (df_featured["closed_date"] > SNAPSHOT_DATE)
        ).sum()
    )
)
print(
    "Rows with coordinates:",
    int(df_featured["has_coordinates"].sum())
)

Shape: (348180, 23)
Valid closure records: 345484
Invalid/unavailable closure: 2696
Future closed_date excluded: 0
Rows with coordinates: 345075


In [27]:
print(
    df_featured[
        [
            "created_date",
            "closed_date",
            "closure_valid",
            "closure_hours",
            "year",
            "month",
            "weekday",
            "hour",
            "is_weekend"
        ]
    ].head(10)
)

print(
    "\nNegative closure hours:",
    (
        df_featured["closure_hours"] < 0
    ).sum()
)

         created_date         closed_date  closure_valid  closure_hours  year  \
0 2025-01-01 00:00:12 2025-01-02 17:05:54           True      41.095000  2025   
1 2025-01-01 00:00:13 2025-01-01 00:05:19           True       0.085000  2025   
2 2025-01-01 00:00:17 2025-01-01 09:01:06           True       9.013611  2025   
3 2025-01-01 00:00:19 2025-01-01 00:43:08           True       0.713611  2025   
4 2025-01-01 00:00:34 2025-01-02 12:04:30           True      36.065556  2025   
5 2025-01-01 00:00:37 2025-01-01 22:50:30           True      22.831389  2025   
6 2025-01-01 00:00:42 2025-01-01 00:41:12           True       0.675000  2025   
7 2025-01-01 00:00:50 2025-01-01 01:46:05           True       1.754167  2025   
8 2025-01-01 00:00:50 2025-01-02 17:09:55           True      41.151389  2025   
9 2025-01-01 00:01:03 2025-01-01 08:38:08           True       8.618056  2025   

   month    weekday  hour  is_weekend  
0      1  Wednesday     0       False  
1      1  Wednesday     0   

In [28]:
print("=== FINAL VALIDATION - JAN 2025 ===")

print("Rows:", len(df_featured))
print(
    "Duplicate unique_key:",
    df_featured["unique_key"].duplicated().sum()
)
print(
    "Missing created_date:",
    df_featured["created_date"].isna().sum()
)
print(
    "Negative closure_hours:",
    (df_featured["closure_hours"] < 0).sum()
)

future_marked_valid = (
    df_featured["closure_valid"]
    & (df_featured["closed_date"] > SNAPSHOT_DATE)
).sum()

print(
    "Future closed_date marked valid:",
    int(future_marked_valid)
)

assert future_marked_valid == 0

print("Years:", df_featured["year"].unique())
print("Months:", df_featured["month"].unique())

print("\nClosure time distribution:")
display(
    df_featured["closure_hours"].describe(
        percentiles=[
            0.5,
            0.75,
            0.9,
            0.95,
            0.99
        ]
    )
)

=== FINAL VALIDATION - JAN 2025 ===
Rows: 348180
Duplicate unique_key: 0
Missing created_date: 0
Negative closure_hours: 0
Future closed_date marked valid: 0
Years: [2025]
Months: [1]

Closure time distribution:


count    345484.000000
mean        239.119432
std         864.136044
min           0.000000
50%          15.093333
75%          68.538125
90%         395.071389
95%        1440.470458
99%        4960.088008
max       15371.430000
Name: closure_hours, dtype: float64

In [29]:
def process_partition(df):
    """Clean one raw partition and create analysis features."""
    df = clean_partition(df)

    # -------------------------
    # Time features
    # -------------------------
    df["year"] = df["created_date"].dt.year
    df["month"] = df["created_date"].dt.month
    df["weekday"] = df["created_date"].dt.day_name()
    df["hour"] = df["created_date"].dt.hour
    df["is_weekend"] = (
        df["created_date"].dt.dayofweek >= 5
    )

    # -------------------------
    # Closure validity
    # -------------------------
    df["closure_valid"] = (
        df["created_date"].notna()
        & df["closed_date"].notna()
        & (df["closed_date"] >= df["created_date"])
        & (df["closed_date"] <= SNAPSHOT_DATE)
    )

    df["closure_hours"] = np.nan
    valid_mask = df["closure_valid"]

    df.loc[
        valid_mask,
        "closure_hours"
    ] = (
        (
            df.loc[valid_mask, "closed_date"]
            - df.loc[valid_mask, "created_date"]
        )
        .dt.total_seconds()
        / 3600
    )

    df["closure_hours"] = pd.to_numeric(
        df["closure_hours"],
        errors="coerce"
    )

    # -------------------------
    # Coordinate availability
    # -------------------------
    df["has_coordinates"] = (
        df["latitude"].notna()
        & df["longitude"].notna()
    )

    return df

In [30]:
test_raw = pd.read_parquet(
    RAW_DIR /
    "nyc311_2025_01_raw.parquet"
)

test_processed = process_partition(
    test_raw
)

print(
    "Raw shape      :",
    test_raw.shape
)

print(
    "Processed shape:",
    test_processed.shape
)

print(
    "Duplicates:",
    test_processed["unique_key"]
    .duplicated()
    .sum()
)

print(
    "Negative closure:",
    (
        test_processed["closure_hours"] < 0
    ).sum()
)

future_valid_test = (
    test_processed["closure_valid"]
    & (test_processed["closed_date"] > SNAPSHOT_DATE)
).sum()

print(
    "Future closed_date marked valid:",
    int(future_valid_test)
)
assert future_valid_test == 0

Raw shape      : (348180, 15)
Processed shape: (348180, 23)
Duplicates: 0
Negative closure: 0
Future closed_date marked valid: 0


## 5. Xử lý và ghi lại 48 partition

Do quy tắc `closure_valid` đã thay đổi, notebook đặt `FORCE_REBUILD_PROCESSED = True` để **ghi đè 48 file processed hiện có trong lần chạy cập nhật này**. Sau khi rebuild thành công và lưu notebook, có thể đổi biến này thành `False` để lần chạy sau bỏ qua file đã có.

In [31]:
FORCE_REBUILD_PROCESSED = True

import gc
import time

processing_log = []

for i, raw_file in enumerate(
    raw_files,
    start=1
):

    output_name = (
        raw_file.name
        .replace(
            "_raw.parquet",
            "_processed.parquet"
        )
    )

    output_file = (
        PROCESSED_DIR
        / output_name
    )

    # Resume:
    # file đã xử lý thì bỏ qua
    if output_file.exists() and not FORCE_REBUILD_PROCESSED:
        print(
            f"[{i:02d}/48] "
            f"SKIP {output_name}"
        )
        continue

    print()
    print(
        f"[{i:02d}/48] "
        f"Processing {raw_file.name}"
    )

    start = time.perf_counter()

    # -------------------------
    # Read
    # -------------------------
    raw = pd.read_parquet(
        raw_file
    )

    raw_rows = len(raw)

    # -------------------------
    # Clean + features
    # -------------------------
    processed = process_partition(
        raw
    )

    # -------------------------
    # Validation
    # -------------------------

    if len(processed) != raw_rows:
        raise ValueError(
            f"Row count changed: "
            f"{raw_file.name}"
        )

    duplicates = (
        processed["unique_key"]
        .duplicated()
        .sum()
    )

    if duplicates != 0:
        raise ValueError(
            f"Duplicate unique_key "
            f"in {raw_file.name}: "
            f"{duplicates}"
        )

    if (
        processed["created_date"]
        .isna()
        .sum()
        != 0
    ):
        raise ValueError(
            f"Missing created_date "
            f"in {raw_file.name}"
        )

    negative_closure = (
        processed["closure_hours"]
        < 0
    ).sum()

    if negative_closure != 0:
        raise ValueError(
            f"Negative closure_hours "
            f"in {raw_file.name}"
        )

    # -------------------------
    # Save
    # -------------------------

    processed.to_parquet(
        output_file,
        index=False,
        compression="snappy"
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    processing_log.append({
        "file": raw_file.name,
        "rows": len(processed),
        "valid_closure": int(
            processed[
                "closure_valid"
            ].sum()
        ),
        "with_coordinates": int(
            processed[
                "has_coordinates"
            ].sum()
        ),
        "seconds": round(
            elapsed,
            2
        )
    })

    print(
        f"Saved → {output_name}"
    )

    print(
        f"Rows: {len(processed):,} | "
        f"Time: {elapsed:.2f}s"
    )

    # -------------------------
    # Free RAM
    # -------------------------

    del raw
    del processed

    gc.collect()


[01/48] Processing nyc311_2022_01_raw.parquet
Saved → nyc311_2022_01_processed.parquet
Rows: 279,281 | Time: 2.99s

[02/48] Processing nyc311_2022_02_raw.parquet
Saved → nyc311_2022_02_processed.parquet
Rows: 244,234 | Time: 2.61s

[03/48] Processing nyc311_2022_03_raw.parquet
Saved → nyc311_2022_03_processed.parquet
Rows: 251,933 | Time: 2.53s

[04/48] Processing nyc311_2022_04_raw.parquet
Saved → nyc311_2022_04_processed.parquet
Rows: 236,083 | Time: 2.39s

[05/48] Processing nyc311_2022_05_raw.parquet
Saved → nyc311_2022_05_processed.parquet
Rows: 276,664 | Time: 2.77s

[06/48] Processing nyc311_2022_06_raw.parquet
Saved → nyc311_2022_06_processed.parquet
Rows: 271,730 | Time: 2.83s

[07/48] Processing nyc311_2022_07_raw.parquet
Saved → nyc311_2022_07_processed.parquet
Rows: 294,474 | Time: 2.82s

[08/48] Processing nyc311_2022_08_raw.parquet
Saved → nyc311_2022_08_processed.parquet
Rows: 269,234 | Time: 1.11s

[09/48] Processing nyc311_2022_09_raw.parquet
Saved → nyc311_2022_09_pr

## 6. Validation toàn bộ dữ liệu processed

Các kiểm tra dưới đây xác nhận đủ 48 file, không đổi số dòng, schema đúng, không có `closure_hours` âm, cờ tọa độ nhất quán và đặc biệt **không có future `closed_date` nào bị đánh dấu `closure_valid=True`**.

In [32]:
#ktra đầy đủ 48 file processed
import pandas as pd
import pyarrow.parquet as pq

processed_files = sorted(
    PROCESSED_DIR.glob(
        "nyc311_*_*_processed.parquet"
    )
)

print(
    "Processed files:",
    len(processed_files)
)

Processed files: 48


In [33]:
# kiểm tra tổng dòng và từng năm

records = []

for file in processed_files:

    parts = file.stem.split("_")

    year = int(parts[1])
    month = int(parts[2])

    rows = (
        pq.ParquetFile(file)
        .metadata
        .num_rows
    )

    records.append({
        "file": file.name,
        "year": year,
        "month": month,
        "rows": rows
    })

processed_summary = pd.DataFrame(records)

year_summary = (
    processed_summary
    .groupby("year")
    .agg(
        files=("file", "count"),
        rows=("rows", "sum")
    )
    .reset_index()
)

display(year_summary)

print(
    "TOTAL:",
    f"{processed_summary['rows'].sum():,}"
)

,year,files,rows
0,2022,12,3169960
1,2023,12,3224723
2,2024,12,3456770
3,2025,12,3655041


TOTAL: 13,506,494


In [34]:
# so từnng file raw với processed 

comparison = []

for processed_file in processed_files:

    parts = processed_file.stem.split("_")

    year = int(parts[1])
    month = int(parts[2])

    raw_file = (
        RAW_DIR
        / f"nyc311_{year}_{month:02d}_raw.parquet"
    )

    raw_rows = (
        pq.ParquetFile(raw_file)
        .metadata
        .num_rows
    )

    processed_rows = (
        pq.ParquetFile(processed_file)
        .metadata
        .num_rows
    )

    comparison.append({
        "year": year,
        "month": month,
        "raw_rows": raw_rows,
        "processed_rows": processed_rows,
        "difference": (
            processed_rows - raw_rows
        )
    })

comparison_df = pd.DataFrame(comparison)

display(
    comparison_df[
        comparison_df["difference"] != 0
    ]
)

print(
    "Files with row mismatch:",
    (
        comparison_df["difference"] != 0
    ).sum()
)

,year,month,raw_rows,processed_rows,difference


Files with row mismatch: 0


In [35]:
# tạo thêm 8 features cho mục đích nghiên cứu

EXPECTED_PROCESSED_COLUMNS = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "community_board",
    "borough",
    "status",
    "open_data_channel_type",
    "latitude",
    "longitude",

    "year",
    "month",
    "weekday",
    "hour",
    "is_weekend",
    "closure_valid",
    "closure_hours",
    "has_coordinates"
]

expected_set = set(
    EXPECTED_PROCESSED_COLUMNS
)

schema_errors = []

for file in processed_files:

    actual_columns = (
        pq.read_schema(file)
        .names
    )

    actual_set = set(actual_columns)

    missing = (
        expected_set - actual_set
    )

    extra = (
        actual_set - expected_set
    )

    if missing or extra:
        schema_errors.append({
            "file": file.name,
            "missing": sorted(missing),
            "extra": sorted(extra)
        })

print(
    "Files with schema error:",
    len(schema_errors)
)

if schema_errors:
    display(
        pd.DataFrame(schema_errors)
    )

Files with schema error: 0


In [36]:
validation_results = []

for i, file in enumerate(
    processed_files,
    start=1
):
    parts = file.stem.split("_")
    expected_year = int(parts[1])
    expected_month = int(parts[2])

    part = pd.read_parquet(
        file,
        columns=[
            "unique_key",
            "created_date",
            "closed_date",
            "closure_valid",
            "closure_hours",
            "latitude",
            "longitude",
            "has_coordinates"
        ]
    )

    duplicates = part["unique_key"].duplicated().sum()
    missing_created = part["created_date"].isna().sum()

    wrong_partition = (
        (part["created_date"].dt.year != expected_year)
        | (part["created_date"].dt.month != expected_month)
    ).sum()

    negative_closure = (
        part["closure_hours"] < 0
    ).sum()

    expected_closure_valid = (
        part["created_date"].notna()
        & part["closed_date"].notna()
        & (part["closed_date"] >= part["created_date"])
        & (part["closed_date"] <= SNAPSHOT_DATE)
    )

    closure_rule_error = (
        part["closure_valid"]
        != expected_closure_valid
    ).sum()

    closure_value_error = (
        (
            part["closure_valid"]
            & part["closure_hours"].isna()
        )
        | (
            ~part["closure_valid"]
            & part["closure_hours"].notna()
        )
    ).sum()

    future_marked_valid = (
        part["closure_valid"]
        & (part["closed_date"] > SNAPSHOT_DATE)
    ).sum()

    expected_coordinates = (
        part["latitude"].notna()
        & part["longitude"].notna()
    )

    coordinate_flag_error = (
        part["has_coordinates"]
        != expected_coordinates
    ).sum()

    validation_results.append({
        "file": file.name,
        "duplicates": int(duplicates),
        "missing_created": int(missing_created),
        "wrong_partition": int(wrong_partition),
        "negative_closure": int(negative_closure),
        "closure_rule_error": int(closure_rule_error),
        "closure_value_error": int(closure_value_error),
        "future_marked_valid": int(future_marked_valid),
        "coordinate_flag_error": int(coordinate_flag_error)
    })

    print(
        f"[{i:02d}/48] "
        f"{file.name} checked"
    )

validation_df = pd.DataFrame(validation_results)

[01/48] nyc311_2022_01_processed.parquet checked
[02/48] nyc311_2022_02_processed.parquet checked
[03/48] nyc311_2022_03_processed.parquet checked
[04/48] nyc311_2022_04_processed.parquet checked
[05/48] nyc311_2022_05_processed.parquet checked
[06/48] nyc311_2022_06_processed.parquet checked
[07/48] nyc311_2022_07_processed.parquet checked
[08/48] nyc311_2022_08_processed.parquet checked
[09/48] nyc311_2022_09_processed.parquet checked
[10/48] nyc311_2022_10_processed.parquet checked
[11/48] nyc311_2022_11_processed.parquet checked
[12/48] nyc311_2022_12_processed.parquet checked
[13/48] nyc311_2023_01_processed.parquet checked
[14/48] nyc311_2023_02_processed.parquet checked
[15/48] nyc311_2023_03_processed.parquet checked
[16/48] nyc311_2023_04_processed.parquet checked
[17/48] nyc311_2023_05_processed.parquet checked
[18/48] nyc311_2023_06_processed.parquet checked
[19/48] nyc311_2023_07_processed.parquet checked
[20/48] nyc311_2023_08_processed.parquet checked
[21/48] nyc311_2023_

In [37]:
validation_columns = [
    "duplicates",
    "missing_created",
    "wrong_partition",
    "negative_closure",
    "closure_rule_error",
    "closure_value_error",
    "future_marked_valid",
    "coordinate_flag_error"
]

validation_totals = (
    validation_df[
        validation_columns
    ]
    .sum()
)

display(validation_totals)

assert (
    validation_totals == 0
).all(), "Processed validation still contains errors."

validation_path = (
    PROCESSED_DIR
    / "processed_validation_log.csv"
)

validation_df.to_csv(
    validation_path,
    index=False
)

print("Saved:", validation_path)

duplicates               0
missing_created          0
wrong_partition          0
negative_closure         0
closure_rule_error       0
closure_value_error      0
future_marked_valid      0
coordinate_flag_error    0
dtype: int64

Saved: d:\nyc311_analysis\data\processed\processed_validation_log.csv


In [38]:
# ktra cleaning thực sự áp dụng cho toàn 4 năm

checks = {
    "descriptor_NA": 0,
    "descriptor_Unspecified": 0,
    "location_type_NA": 0,
    "location_type_Unspecified": 0,
    "borough_Unspecified": 0,
    "status_Unspecified": 0,
    "channel_UNKNOWN": 0
}

for file in processed_files:

    part = pd.read_parquet(
        file,
        columns=[
            "descriptor",
            "location_type",
            "borough",
            "status",
            "open_data_channel_type"
        ]
    )

    checks["descriptor_NA"] += int(
        (part["descriptor"] == "N/A").sum()
    )

    checks["descriptor_Unspecified"] += int(
        (
            part["descriptor"]
            == "Unspecified"
        ).sum()
    )

    checks["location_type_NA"] += int(
        (
            part["location_type"]
            == "N/A"
        ).sum()
    )

    checks[
        "location_type_Unspecified"
    ] += int(
        (
            part["location_type"]
            == "Unspecified"
        ).sum()
    )

    checks["borough_Unspecified"] += int(
        (
            part["borough"]
            == "Unspecified"
        ).sum()
    )

    checks["status_Unspecified"] += int(
        (
            part["status"]
            == "Unspecified"
        ).sum()
    )

    checks["channel_UNKNOWN"] += int(
        (
            part["open_data_channel_type"]
            == "UNKNOWN"
        ).sum()
    )

cleaning_check = pd.Series(checks)

display(cleaning_check)

descriptor_NA                      0
descriptor_Unspecified             0
location_type_NA                   0
location_type_Unspecified          0
borough_Unspecified                0
status_Unspecified                 0
channel_UNKNOWN              1072807
dtype: int64

## Kết luận bước làm sạch

- Giữ nguyên toàn bộ request thay vì xóa dòng chỉ vì thiếu descriptor, vị trí hoặc `closed_date`; từng phân tích sẽ chọn subset phù hợp.
- `UNKNOWN` ở kênh gửi được giữ như một category quan sát được.
- `closure_hours` chỉ tồn tại khi closure hợp lệ theo **ngày tạo ≤ ngày đóng ≤ 06/10/2026**.
- Sau khi chạy notebook này, cần chạy lại notebook EDA để cập nhật các bảng closure cache theo quy tắc mới.